In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

if !isdefined(Main, :nb_paths)
    include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))
end

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "hydrogen_1d", "vmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

if !isdefined(Main, :System1D)
    include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
end
using .System1D

default(; dpi=170)
nothing


## Model and VMC Parameters

This notebook compares two VMC proposal kernels for the regularized one-dimensional Coulomb potential
`V(x) = -1 / sqrt(x^2 + s^2)` with softening `s = 0.5`.

The shared trial state is
`log |psi_T(x)| = -alpha * sqrt(x^2 + s^2)`.

Parameters used below:
- Softening `s = 0.5`
- Trial parameter `alpha = 1.0`
- Time step `dt = 2.0e-2`
- Total steps `nsteps = 180`
- Walker count `targetN = 12000`
- Burn-in used in the summary `ANALYSIS_BURN_IN = 36`

Proposal comparison:
- `DriftGaussianProposal()`
- `GaussianProposal()`


## Julia Construction

The next cell defines the soft-Coulomb Hamiltonian, the trial state, the proposal comparison setup, and the notebook toggles.

That cell is also where the CSV filename and debug cadence are set.


In [ ]:
softening = 0.5
alpha = 1.0

V(R) = -1 / sqrt(R[1]^2 + softening^2)
H = Hamiltonian(1, 0.5, V)

logpsi(R) = begin
    x = R[1]
    r = sqrt(x^2 + softening^2)
    -alpha * r
end
gradlogpsi(R) = begin
    x = R[1]
    r = sqrt(x^2 + softening^2)
    [-alpha * x / r]
end
lapllogpsi(R) = begin
    x = R[1]
    r = sqrt(x^2 + softening^2)
    -alpha * softening^2 / r^3
end
trial = TrialWF(logpsi, gradlogpsi, lapllogpsi)

targetN = 12000
dt = 2.0e-2
nsteps = 180
ET0 = -0.5
ANALYSIS_BURN_IN = 36

params = VMCParams(; dt=dt, nsteps=nsteps, targetN=targetN, ET0=ET0)

rng_init = MersenneTwister(1234)
base_positions = [[randn(rng_init)] for _ in 1:targetN]

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
NBINS = 180
DENSITY_SMOOTHING = 9
DENSITY_RANGE = (-8.0, 8.0)
X_AXIS_LABEL = "x"

PLOT_TITLE = "Regularized hydrogen VMC"
DENSITY_TITLE = "Regularized hydrogen VMC: final density comparison"
RUN_LABELS = ["drifted Gaussian", "Gaussian"]
RUN_COLORS = [:navy, :darkorange]

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 10
WRITE_RUN_CSV = false
CSV_FILENAME = "hydrogen_1d_regularized_vmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "hydrogen_1d_regularized_vmc"


In [ ]:
sim_drift = run_vmc(
    H,
    params,
    base_positions,
    trial;
    rng=MersenneTwister(42),
    proposal=DriftGaussianProposal(),
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABELS[1],
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

sim_gauss = run_vmc(
    H,
    params,
    base_positions,
    trial;
    rng=MersenneTwister(43),
    proposal=GaussianProposal(),
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABELS[2],
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

for (label, sim_ref) in zip(RUN_LABELS, (sim_drift, sim_gauss))
    start_idx = min(ANALYSIS_BURN_IN + 1, length(sim_ref.energy_history))
    mean_energy, sem_energy = nb_mean_sem(sim_ref.energy_history[start_idx:end])
    println(@sprintf("%s mean energy after burn-in=%d: %.8f +/- %.3e", label, ANALYSIS_BURN_IN, mean_energy, sem_energy))
    println(@sprintf("%s final acceptance rate = %.4f", label, sim_ref.acceptance_rate))
end

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    rows = vcat(nb_vmc_rows(RUN_LABELS[1], sim_drift), nb_vmc_rows(RUN_LABELS[2], sim_gauss))
    nb_write_csv(csv_path, rows)
    println("Wrote run CSV to: ", abspath(csv_path))
end

SIMS = [sim_drift, sim_gauss]
SIM_LABELS = RUN_LABELS
SIM_COLORS = RUN_COLORS


In [ ]:
history_fig = nb_plot_vmc_history(SIMS; labels=SIM_LABELS, colors=SIM_COLORS, title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

if DENSITY_RANGE === nothing
    coord_values = Float64[]
    for sim in SIMS
        append!(coord_values, nb_all_coordinates(sim; coord=1))
    end
    xlo, xhi = nb_padded_limits(coord_values; pad_frac=0.12)
else
    xlo, xhi = DENSITY_RANGE
end

density_fig = plot(
    xlabel=X_AXIS_LABEL,
    ylabel="density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(xlo, xhi),
)

for (sim, label, color) in zip(SIMS, SIM_LABELS, SIM_COLORS)
    centers, density = nb_density_curve_from_snapshot(
        nb_last_snapshot(sim);
        coord=1,
        nbins=NBINS,
        xmin=xlo,
        xmax=xhi,
        smoothing_window=DENSITY_SMOOTHING,
    )
    plot!(density_fig, centers, density; label=label, color=color, linewidth=2.4)
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)
